In [1]:
from datasets import Dataset, load_dataset
from glob import glob
import json

/home/ubuntu/synthesize-data/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_data = load_dataset("phunguyen01/r1-sft", split="train")

len(base_data)

237432

In [3]:
base_dir = "results_tag/r1-sft"

quality_files = glob(f"{base_dir}/quality_*.json")
classification_files = glob(f"{base_dir}/classification_*.json")

In [4]:
quality_files

['results_tag/r1-sft/quality_start-118716_offset-29679.json',
 'results_tag/r1-sft/quality_start-148395_offset-29679.json',
 'results_tag/r1-sft/quality_start-0_offset-29679.json',
 'results_tag/r1-sft/quality_start-29679_offset-29679.json',
 'results_tag/r1-sft/quality_start-198074_offset-None.json',
 'results_tag/r1-sft/quality_start-178074_offset-20000.json',
 'results_tag/r1-sft/quality_start-89037_offset-29679.json',
 'results_tag/r1-sft/quality_start-59358_offset-29679.json']

In [5]:
def get_start_from_path(path):
    # Extract the start value from filename like quality_start-118716_offset-29679.json
    start = path.split("start-")[1].split("_")[0]
    return int(start)

# sort the files by start value
quality_files.sort(key=get_start_from_path)
classification_files.sort(key=get_start_from_path)

In [6]:
quality_data = []

for file in quality_files:
    data = json.load(open(file))
    quality_data.extend(data)
    
task_domain_data = []
for file in classification_files:
    data = json.load(open(file))
    task_domain_data.extend(data)

In [7]:
[i for i in quality_data if i["quality"]=="very poor"][0]

{'instruction': 'What are the reasons behind denying the starry night skies the right to own a business?',
 'quality': 'very poor',
 'quality_explanation': "The query is quite unclear and lacks context. It is not clear what is meant by 'denying the starry night skies the right to own a business.' The phrase 'starry night skies' is metaphorical and does not typically refer to an entity capable of owning a business. The query could benefit from more specific details or context to clarify the intent behind the question.",
 'quality_generator': 'Qwen/Qwen2.5-14B-Instruct'}

In [16]:
new_data = []
for i in range(len(base_data)):
    assert base_data[i]["problem"] == quality_data[i]["instruction"]
    assert base_data[i]["problem"] == task_domain_data[i]["instruction"]
    
    new_data.append({
        **base_data[i],
        "quality": quality_data[i]["quality"],
        "quality_explanation": quality_data[i]["quality_explanation"] if quality_data[i]["quality_explanation"] else "Error",
        "quality_generator": quality_data[i]["quality_generator"],
        "task_category": task_domain_data[i]["task_category"],
        "other_task_category": task_domain_data[i]["other_task_category"],
        "task_category_generator": task_domain_data[i]["task_category_generator"]
    })


In [17]:
final_data = Dataset.from_list(new_data)